### Find neighbours of atoms in PDB structures 

The PDB structure contains the (x, y, z) coordinates of every atom in the structure. We will now use the implementation of the  algorithm, `NeighborSearch`, from biopython that can find neighbouring atoms that are within a certain distance of a given coordinate.

In [5]:
import os
from Bio.PDB import PDBParser, NeighborSearch

pfad = os.path.join("daten", "output.txt")
if os.path.exists(pfad):
    print("Datei vorhanden:", pfad)

# this is the directory where our PDB files that we want to use are stored. it will here be called PDB_DIR and a a small reminder rn we are still working with only the corona files 

PDB_DIR = "../data/pdbs_corona_cnf"

# now we neeed to find the path to the corresponding PDB File 

pdb_id = "6xc3"
filename = os.path.join(PDB_DIR, f"{pdb_id}.pdb")


first we will parse the PDB file

In [6]:
from Bio.PDB import PDBParser

parser = PDBParser(PERMISSIVE=1)
structure = parser.get_structure(pdb_id, filename)

Then we need to initialise the data structure that is used to perform the search of neighbours. Here we use all atoms of the structure for demonstration. This can be wasteful. Later, it can be advantageous to only use the atoms of the antigen, for example.

In [7]:
atoms = list(structure.get_atoms())
ns = NeighborSearch(atoms)

Now we can use the `ns.search(coords, distance)` method to find all atoms that are within `distance` angstroms from the position specified by the `[x, y, z]` array `coords`.

We can retrieve the position of an atom using its `coord` property:

In [8]:
atom = structure[0]['H'][52]['N']
atom.coord

array([-60.134,  62.357,  23.903], dtype=float32)

To find all atoms in vicinity of an atom we simply pass the coordinates of the atom to the `ns.search` method.

In [10]:
close_atoms = ns.search(atom.coord, 4.0)
close_atoms

[<Atom N>,
 <Atom O>,
 <Atom CD>,
 <Atom CE3>,
 <Atom CG>,
 <Atom CB>,
 <Atom CA>,
 <Atom C>,
 <Atom CG2>,
 <Atom N>,
 <Atom CA>,
 <Atom CB>,
 <Atom CG1>,
 <Atom O>,
 <Atom N>,
 <Atom C>,
 <Atom O>]

i think at this point its useful to do some sensitivity testing 

`close_atoms` now contains the atom entities of all atoms that are within 4.0 angstrom of the supplied coordinates. To find the residues and chains these atoms belong to, we can either use repeated application of the `get_parent()` method, or we can simply use the `get_full_id()` method.

As we want the output as a DataFrame, we first create a vector of dicts, and then create the DataFrame. 